In [ ]:
"""
=============================================================================
SAM (SEGMENT ANYTHING MODEL) PAPER DETECTION PIPELINE
=============================================================================

INSTALLATION INSTRUCTIONS:
--------------------------
Run these commands in your terminal with UV:

    uv pip install segment-anything
    uv pip install git+https://github.com/facebookresearch/segment-anything.git

Or just:
    uv pip install segment-anything

WHAT IS SAM:
------------
- Meta's Segment Anything Model (SAM)
- State-of-the-art segmentation model
- Can segment ANY object with minimal prompting
- More reliable than U²-Net for paper detection
- Works with points, boxes, or automatic segmentation

WHY SAM IS BETTER FOR YOUR CASE:
---------------------------------
✅ More robust to different backgrounds
✅ Better edge detection (will find the 4 corners correctly)
✅ Handles occlusions (hands, shadows) better
✅ No training needed - works out of the box
✅ Can use "automatic everything mode" to find paper

MODEL SIZES:
------------
- vit_h (huge): 2.4GB - Best accuracy
- vit_l (large): 1.2GB - Good balance
- vit_b (base): 375MB - Fastest, still accurate

=============================================================================
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor
import urllib.request
from pathlib import Path


# =============================================================================
# STEP 1: DOWNLOAD AND LOAD SAM MODEL
# =============================================================================

class SAM_PaperDetector:
    """
    SAM-based paper detector
    Uses Meta's Segment Anything Model for robust paper segmentation
    """

    def __init__(self, model_type="vit_b"):
        """
        Initialize SAM model

        Args:
            model_type: 'vit_b' (375MB), 'vit_l' (1.2GB), or 'vit_h' (2.4GB)
                       vit_b is recommended - fast and accurate enough
        """
        print(f"🔄 Initializing SAM model ({model_type})...")

        self.model_type = model_type
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"   Device: {self.device}")

        # Download checkpoint if not exists
        checkpoint_path = self._download_checkpoint(model_type)

        # Load SAM model
        print("📂 Loading SAM model...")
        self.sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
        self.sam.to(device=self.device)

        # Create mask generator for automatic segmentation
        # This will find ALL objects in the image automatically
        self.mask_generator = SamAutomaticMaskGenerator(
            model=self.sam,
            points_per_side=32,  # More points = better accuracy, slower
            pred_iou_thresh=0.88,  # Only keep high-quality masks
            stability_score_thresh=0.95,  # Only keep stable masks
            crop_n_layers=1,  # Crop and re-segment for better results
            crop_n_points_downscale_factor=2,
            min_mask_region_area=1000,  # Ignore tiny regions
        )

        print("✓ SAM model ready!")

    def _download_checkpoint(self, model_type):
        """
        Download SAM checkpoint if not exists

        Returns:
            Path to checkpoint file
        """
        # Model URLs from Meta's SAM repository
        checkpoint_urls = {
            'vit_b': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
            'vit_l': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth',
            'vit_h': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
        }

        # Create cache directory
        cache_dir = Path.home() / ".cache" / "sam_models"
        cache_dir.mkdir(parents=True, exist_ok=True)

        checkpoint_path = cache_dir / f"sam_{model_type}.pth"

        # Download if not exists
        if not checkpoint_path.exists():
            print(f"📥 Downloading SAM {model_type} checkpoint...")
            print(f"   URL: {checkpoint_urls[model_type]}")
            print(f"   This may take a few minutes...")

            urllib.request.urlretrieve(
                checkpoint_urls[model_type],
                checkpoint_path,
                reporthook=self._download_progress
            )
            print(f"\n✓ Downloaded to {checkpoint_path}")
        else:
            print(f"✓ Using cached checkpoint: {checkpoint_path}")

        return str(checkpoint_path)

    def _download_progress(self, block_num, block_size, total_size):
        """Show download progress"""
        downloaded = block_num * block_size
        percent = min(downloaded / total_size * 100, 100)
        print(f"\r   Progress: {percent:.1f}%", end='', flush=True)

    def detect_paper_automatic(self, img):
        """
        Automatically detect paper using SAM's "segment everything" mode

        Args:
            img: RGB image (numpy array)

        Returns:
            mask: Binary mask of detected paper
        """
        print("🔍 Running SAM automatic segmentation...")

        # Generate all masks in the image
        masks = self.mask_generator.generate(img)

        if len(masks) == 0:
            raise RuntimeError("❌ No objects detected by SAM")

        print(f"   Found {len(masks)} objects")

        # Sort by area (paper should be largest object)
        masks_sorted = sorted(masks, key=lambda x: x['area'], reverse=True)

        # Get largest mask (assumed to be paper)
        paper_mask = masks_sorted[0]['segmentation']

        # Convert boolean mask to uint8
        mask = (paper_mask * 255).astype(np.uint8)

        print(f"✓ Paper detected (area: {masks_sorted[0]['area']} pixels)")

        return mask, masks_sorted

    def detect_paper_with_box(self, img, box):
        """
        Detect paper using bounding box prompt
        Useful if you know approximate paper location

        Args:
            img: RGB image
            box: [x1, y1, x2, y2] bounding box

        Returns:
            mask: Binary mask of detected paper
        """
        print("🔍 Running SAM with box prompt...")

        predictor = SamPredictor(self.sam)
        predictor.set_image(img)

        # Predict mask using box prompt
        masks, scores, logits = predictor.predict(
            box=np.array(box),
            multimask_output=False  # Return single best mask
        )

        mask = (masks[0] * 255).astype(np.uint8)

        print(f"✓ Paper detected with box prompt (score: {scores[0]:.3f})")

        return mask


# =============================================================================
# STEP 2: SEGMENT PAPER USING SAM MASK
# =============================================================================

def segment_paper_from_mask(img, mask, debug=False):
    """
    Extract paper region using SAM mask

    Args:
        img: Original RGB image
        mask: Binary mask from SAM
        debug: Show visualization

    Returns:
        Cropped paper image with white background
    """

    # --- Find paper contour ---
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        raise RuntimeError("❌ No contours found in SAM mask")

    # Get largest contour (paper)
    largest_contour = max(contours, key=cv2.contourArea)

    # Get bounding rectangle
    x, y, w, h = cv2.boundingRect(largest_contour)

    # Add padding
    pad = 10
    x = max(0, x - pad)
    y = max(0, y - pad)
    w = min(img.shape[1] - x, w + 2 * pad)
    h = min(img.shape[0] - y, h + 2 * pad)

    # Crop image and mask
    cropped_img = img[y:y + h, x:x + w].copy()
    cropped_mask = mask[y:y + h, x:x + w]

    # --- Apply mask with white background ---
    white_bg = np.full_like(cropped_img, 255, dtype=np.uint8)
    result = np.where(cropped_mask[..., None] == 255, cropped_img, white_bg)

    if debug:
        plt.figure(figsize=(15, 5))

        plt.subplot(131)
        plt.imshow(img)
        plt.title("Original Image")
        plt.axis('off')

        plt.subplot(132)
        plt.imshow(mask, cmap='gray')
        plt.title("SAM Mask")
        plt.axis('off')

        plt.subplot(133)
        plt.imshow(result)
        plt.title("Segmented Paper")
        plt.axis('off')

        plt.tight_layout()
        plt.show()

    return result


# =============================================================================
# STEP 3: VISUALIZE ALL DETECTED OBJECTS (DEBUGGING)
# =============================================================================

def visualize_sam_masks(img, masks_sorted, top_n=5):
    """
    Show top N detected objects from SAM
    Useful for debugging if wrong object is selected

    Args:
        img: Original image
        masks_sorted: Sorted masks from SAM (by area)
        top_n: Number of top masks to show
    """
    n = min(top_n, len(masks_sorted))

    fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4))

    # Show original
    axes[0].imshow(img)
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    # Show top N masks
    for i in range(n):
        mask = masks_sorted[i]['segmentation']
        area = masks_sorted[i]['area']

        axes[i + 1].imshow(mask, cmap='gray')
        axes[i + 1].set_title(f"Rank {i + 1}\nArea: {area:,} px")
        axes[i + 1].axis('off')

    plt.tight_layout()
    plt.show()


# =============================================================================
# STEP 4: NORMALIZE ORIENTATION
# =============================================================================

def normalize_orientation(img):
    """Rotate portrait to landscape"""
    h, w = img.shape[:2]
    if h > w:
        img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
        print(f"   ↻ Rotated from portrait ({w}×{h}) to landscape ({h}×{w})")
    else:
        print(f"   → Already landscape ({w}×{h})")
    return img


# =============================================================================
# STEP 5: CLEAN WHITE BACKGROUND
# =============================================================================

def clean_white_background(img):
    """Make white areas pure white, preserve colored strokes"""
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    # Brighten already-bright pixels (paper background)
    bright_mask = l_channel > 200
    l_channel[bright_mask] = 255

    lab_cleaned = cv2.merge((l_channel, a_channel, b_channel))
    result = cv2.cvtColor(lab_cleaned, cv2.COLOR_LAB2RGB)

    return result


# =============================================================================
# STEP 6: ENHANCE CRAYON COLORS
# =============================================================================

def enhance_crayon_colors(img):
    """Gently boost saturation to make crayons vivid"""
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    h_channel, s_channel, v_channel = cv2.split(hsv)

    # Boost saturation by 15% + small offset
    s_channel = s_channel.astype(np.float32)
    s_channel = s_channel * 1.15 + 5
    s_channel = np.clip(s_channel, 0, 255).astype(np.uint8)

    # Slight brightness boost
    v_channel = v_channel.astype(np.float32)
    v_channel = v_channel * 1.02
    v_channel = np.clip(v_channel, 0, 255).astype(np.uint8)

    hsv_enhanced = cv2.merge((h_channel, s_channel, v_channel))
    result = cv2.cvtColor(hsv_enhanced, cv2.COLOR_HSV2RGB)

    return result


# =============================================================================
# STEP 7: RESIZE TO DATASET FORMAT
# =============================================================================

def resize_to_dataset(img, target_width=512):
    """Resize maintaining aspect ratio"""
    h, w = img.shape[:2]
    scale = target_width / w
    new_width = target_width
    new_height = int(h * scale)

    resized = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_AREA)
    print(f"   📐 Resized: {w}×{h} → {new_width}×{new_height}")

    return resized


# =============================================================================
# STEP 8: COMPLETE PIPELINE WITH SAM
# =============================================================================

def full_pipeline_sam(image_path, detector, debug=False, save_path="output_sam.png"):
    """
    Complete SAM-based processing pipeline

    Args:
        image_path: Path to input image
        detector: SAM_PaperDetector instance
        debug: Show intermediate steps
        save_path: Output file path

    Returns:
        Final processed image
    """

    print(f"\n{'=' * 70}")
    print(f"🎨 Processing: {image_path}")
    print(f"{'=' * 70}")

    # --- LOAD IMAGE ---
    img = cv2.imread(image_path)
    if img is None:
        raise RuntimeError(f"❌ Cannot read image: {image_path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print(f"✓ Loaded image: {img.shape[1]}×{img.shape[0]} pixels")

    # --- STEP 1: Detect Paper with SAM ---
    print("\n🔍 Step 1: Detecting paper with SAM...")
    mask, all_masks = detector.detect_paper_automatic(img)

    # Optional: Show all detected objects for debugging
    if debug:
        print("\n📊 Showing all detected objects...")
        visualize_sam_masks(img, all_masks, top_n=5)

    # --- STEP 2: Segment Paper ---
    print("\n✂️  Step 2: Segmenting paper from background...")
    paper = segment_paper_from_mask(img, mask, debug=debug)
    print(f"✓ Paper extracted: {paper.shape[1]}×{paper.shape[0]}")

    # --- STEP 3: Normalize Orientation ---
    print("\n🔄 Step 3: Normalizing orientation...")
    paper = normalize_orientation(paper)

    # --- STEP 4: Clean Background ---
    print("\n🧹 Step 4: Cleaning white background...")
    paper = clean_white_background(paper)
    print("✓ Background cleaned")

    # --- STEP 5: Enhance Colors ---
    print("\n🎨 Step 5: Enhancing crayon colors...")
    paper = enhance_crayon_colors(paper)
    print("✓ Colors enhanced")

    # --- STEP 6: Resize ---
    print("\n📏 Step 6: Resizing to dataset format...")
    final = resize_to_dataset(paper, target_width=512)

    # --- SAVE OUTPUT ---
    print(f"\n💾 Saving output to: {save_path}")
    output_bgr = cv2.cvtColor(final, cv2.COLOR_RGB2BGR)
    cv2.imwrite(save_path, output_bgr)

    print(f"\n{'=' * 70}")
    print("✅ PIPELINE COMPLETE!")
    print(f"{'=' * 70}\n")

    return final


# =============================================================================
# STEP 9: BATCH PROCESSING
# =============================================================================

def process_batch_sam(image_folder, output_folder, detector):
    """
    Process multiple images with SAM

    Args:
        image_folder: Folder with input images
        output_folder: Where to save outputs
        detector: SAM_PaperDetector instance
    """
    from pathlib import Path

    Path(output_folder).mkdir(parents=True, exist_ok=True)

    extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
    image_files = []
    for ext in extensions:
        image_files.extend(Path(image_folder).glob(f'*{ext}'))

    print(f"\n📁 Found {len(image_files)} images to process")

    success_count = 0
    fail_count = 0

    for i, img_path in enumerate(image_files, 1):
        try:
            print(f"\n[{i}/{len(image_files)}] Processing: {img_path.name}")

            output_path = Path(output_folder) / f"sam_{img_path.name}"

            final = full_pipeline_sam(
                image_path=str(img_path),
                detector=detector,
                debug=False,
                save_path=str(output_path)
            )

            success_count += 1

        except Exception as e:
            print(f"❌ Failed: {e}")
            fail_count += 1

    print(f"\n{'=' * 70}")
    print(f"✅ Successfully processed: {success_count}")
    print(f"❌ Failed: {fail_count}")
    print(f"{'=' * 70}")


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":

    # --- INITIALIZE SAM DETECTOR (Do this ONCE) ---
    print("🚀 Initializing SAM Paper Detector...")
    print("   Recommended: 'vit_b' for speed (375MB)")
    print("   For best accuracy: 'vit_h' (2.4GB)")

    # Choose model size: 'vit_b', 'vit_l', or 'vit_h'
    detector = SAM_PaperDetector(model_type='vit_b')

    # --- PROCESS SINGLE IMAGE ---
    try:
        final_result = full_pipeline_sam(
            image_path="wood_bg1.jpeg",
            detector=detector,
            debug=True,
            save_path="final_sam_output.png"
        )

        # Display result
        plt.figure(figsize=(10, 7))
        plt.imshow(final_result)
        plt.title("Final SAM Result", fontsize=16, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback

        traceback.print_exc()

# --- BATCH PROCESSING EXAMPLE ---
# detector = SAM_PaperDetector(model_type='vit_b')
# process_batch_sam("input_images/", "output_images/", detector)